In [1]:
# Classificação Automática de Atividades Culturais e Educacionais
# 06 - Experimentos antes do multirrótulo
#
# Integrantes: Beatriz Aparecida de Mello Barbosa (10354067), Bruna Gonçalves Corte David (10425696),
#              Henrique Brainer Costa (10420717), João Pedro Queiroz de Andrade (10425822),
#              Júlia Andrade (10428513)
#
# Conteúdo: passa a limpo, na ordem em que fizemos, os 11 experimentos que usamos para decidir
#           como dar mais de uma categoria a cada atividade e como refazer os grupos sem o teste.
#           Cada experimento termina dizendo o que mostrou e que decisão saiu dele.
#
# Alterações:
#   11/10/2026 - Bruna - notebook novo, com os experimentos que decidiram como fazer a Parte 3

Na Parte 2, cada atividade ficou com uma categoria só, a do grupo em que caiu no notebook 05, o que era
o planejado: a Parte 2 descobria as categorias, e dar mais de uma categoria por atividade ficava para a
Parte 3, junto com o classificador. O próprio artigo da Parte 2 já dizia o que esse passo ia pedir, já
que com uma categoria por atividade o classificador não teria nenhum exemplo de atividade com duas
categorias para aprender, e o parecer pediu o mesmo: exemplos com mais de uma categoria, conferidos por
pessoas. Antes de decidir como montar esses exemplos, fizemos onze experimentos pequenos, e este
notebook refaz todos, na ordem, porque foi deles que saíram as decisões da Parte 3.

Como os experimentos serviram para escolher o caminho, a maioria usa as 693 atividades, enquanto os
números que vão para o artigo saem do notebook 07, feito só com o treino.

In [2]:
# ====================================================================================================
#                       CARREGAMENTO DAS ATIVIDADES, DOS GRUPOS E DOS EMBEDDINGS
# ====================================================================================================
import pandas as pd                                   # leitura e manipulação das tabelas
import numpy as np                                    # contas com vetores e matrizes
import re                                             # separa as descrições em frases
from collections import Counter                       # conta quantas vezes cada coisa aparece
from sklearn.cluster import KMeans                    # o mesmo agrupamento do notebook 05
from sklearn.cluster import AgglomerativeClustering   # agrupa juntando aos poucos as atividades mais parecidas
from sklearn.metrics import adjusted_rand_score       # 1 é agrupar igual, 0 é concordar por acaso

activities = pd.read_csv("../data/03_processed/05_activities_clustered.csv", sep=";", encoding="utf-8-sig")    # atividades com o grupo e a categoria do notebook 05
clusters_summary = pd.read_csv("../data/03_processed/05_clusters_summary.csv", sep=";", encoding="utf-8-sig")  # o nome que demos a cada grupo
activity_embeddings = np.load("../data/03_processed/03_embeddings.npy")                                        # vetores do notebook 03, na mesma ordem das linhas da tabela

print("Atividades:", len(activities), "| Formato dos embeddings:", activity_embeddings.shape)

Atividades: 693 | Formato dos embeddings: (693, 1024)


Todos os experimentos usam a mesma medida: cada categoria ganha um centro, que é a média dos vetores das
atividades dela, e medimos o quanto o vetor de cada atividade aponta para o mesmo lado de cada centro.
Essa medida é a similaridade de cosseno (perto de 1 quer dizer muito parecido, perto de 0 quer dizer que
não tem nada a ver), e no fim cada atividade fica com 13 números, um por categoria.

In [3]:
# ----------------------------------------------------------------------------------------------------
#        A MEDIDA DOS EXPERIMENTOS: SEMELHANÇA DE CADA ATIVIDADE COM O CENTRO DE CADA CATEGORIA
# ----------------------------------------------------------------------------------------------------
embedding_lengths = np.linalg.norm(activity_embeddings, axis=1, keepdims=True)   # comprimento de cada vetor
normalized_embeddings = activity_embeddings / embedding_lengths                  # com comprimento 1, multiplicar dois vetores já dá o cosseno

category_names = sorted(activities["category"].unique())                               # as 13 categorias, em ordem alfabética
category_of_each_activity = activities["category"].to_numpy()                          # categoria que cada atividade recebeu na Parte 2
cluster_of_each_activity = activities["cluster"].values                                # grupo (de 0 a 15) em que cada atividade caiu
name_of_each_cluster = clusters_summary.set_index("cluster")["given_name"].to_dict()   # grupo -> categoria

category_centers = []                                                                                # um centro por categoria
for category_name in category_names:
    embeddings_of_the_category = normalized_embeddings[category_of_each_activity == category_name]   # vetores das atividades da categoria
    category_centers.append(embeddings_of_the_category.mean(axis=0))                                 # o centro é a média deles
category_centers = np.array(category_centers)                                                        # lista vira matriz de 13 linhas
category_centers = category_centers / np.linalg.norm(category_centers, axis=1, keepdims=True)        # centros também com comprimento 1

similarity_to_categories = normalized_embeddings @ category_centers.T   # uma linha por atividade, uma coluna por categoria

print("Categorias:", len(category_names), "| Tabela de semelhanças:", similarity_to_categories.shape)

Categorias: 13 | Tabela de semelhanças: (693, 13)


## Experimento 1. Quanto texto se repete de uma atividade para outra?

No notebook 04, dois grupos do TF-IDF se formaram pelo aviso de inscrição do SESC, e não pelo assunto,
então ficamos com receio de o mesmo texto burocrático estar puxando também os embeddings. Para medir o
tamanho do problema, separamos cada descrição em frases e contamos quantas frases aparecem iguais em 3
atividades ou mais.

In [4]:
# ====================================================================================================
#                        EXPERIMENTO 1: FRASES QUE SE REPETEM ENTRE ATIVIDADES
# ====================================================================================================
def split_into_sentences(description):
    pieces = re.split(r"(?<=[.!?])\s+|\n", str(description))   # quebra depois de ponto, exclamação ou interrogação, e em quebra de linha
    return [piece.strip().lower() for piece in pieces]         # tudo em minúsculas, para a mesma frase não contar como duas

minimum_sentence_length = 25                                                                                                   # frase com até 25 caracteres é curta demais para contar
sentences_of_each_activity = []                                                                                                # as frases de cada atividade, sem repetir dentro dela
for description in activities["description"]:
    long_sentences = [sentence for sentence in split_into_sentences(description) if len(sentence) > minimum_sentence_length]   # só as frases compridas
    sentences_of_each_activity.append(set(long_sentences))                                                                     # o set faz cada frase contar uma vez por atividade

activities_per_sentence = Counter()                             # frase -> em quantas atividades ela aparece
for sentences_of_the_activity in sentences_of_each_activity:
    activities_per_sentence.update(sentences_of_the_activity)   # soma 1 para cada frase da atividade

repeated_sentences = set()                 # frases que aparecem em 3 atividades ou mais
for sentence, number_of_activities in activities_per_sentence.items():
    if number_of_activities >= 3:
        repeated_sentences.add(sentence)   # essa frase é texto repetido

has_repeated_sentence = np.array([len(sentences_of_the_activity & repeated_sentences) > 0 for sentences_of_the_activity in sentences_of_each_activity])   # a atividade tem pelo menos uma frase repetida?

print("Frases repetidas:", len(repeated_sentences))
print("Atividades com pelo menos uma:", has_repeated_sentence.sum(), "de", len(activities))
print("De que instituição elas são:", activities.loc[has_repeated_sentence, "source"].value_counts().to_dict())

Frases repetidas: 47
Atividades com pelo menos uma: 107 de 693
De que instituição elas são: {'SESC': 107}


In [5]:
total_words = 0                                                    # palavras de todas as descrições
words_in_repeated_sentences = 0                                    # palavras que estão dentro de frase repetida
for description in activities["description"]:
    total_words += len(str(description).split())                   # conta as palavras da descrição
    for sentence in split_into_sentences(description):
        if sentence in repeated_sentences:
            words_in_repeated_sentences += len(sentence.split())   # conta as palavras da frase repetida

share_of_repeated_words = 100 * words_in_repeated_sentences / total_words   # quanto da base é texto repetido

print("Palavras da base que estão em frase repetida:", round(share_of_repeated_words, 1), "%")
print()
print("As frases que mais se repetem:")
for sentence, number_of_activities in activities_per_sentence.most_common(5):
    print("  ", number_of_activities, "atividades:", sentence[:110])

Palavras da base que estão em frase repetida: 6.6 %

As frases que mais se repetem:
   24 atividades: caso não sejam preenchidas nos primeiros cinco dias, elas serão abertas ao público em geral.
   19 atividades: você pode acessá-las pelo aplicativocredencial sesc spou pelo sitecentralrelacionamento.sescsp.org.br, na abai
   19 atividades: em seguida, é necessário comparecer à unidade e finalizar o processo nacentral de atendimento, apresentando a 
   17 atividades: as vagas remanescentes são disponibilizadas na primeira e na terceira quinta-feira do mês, a partir das 18h.
   15 atividades: incentiva a autonomia, o prazer pelo esporte e a melhoria da qualidade de vida.


In [6]:
# ----------------------------------------------------------------------------------------------------
#                             EM QUE GRUPOS O TEXTO REPETIDO SE CONCENTRA
# ----------------------------------------------------------------------------------------------------
repeated_per_cluster = pd.DataFrame({"cluster": cluster_of_each_activity, "has_repeated_sentence": has_repeated_sentence})                  # uma linha por atividade
repeated_per_cluster = repeated_per_cluster.groupby("cluster")["has_repeated_sentence"].agg(["sum", "count"])                               # quantas têm frase repetida e quantas são no grupo
repeated_per_cluster.columns = ["with_repeated_sentence", "activities"]                                                                     # nomes mais claros para as duas contagens
repeated_per_cluster["percentage"] = (100 * repeated_per_cluster["with_repeated_sentence"] / repeated_per_cluster["activities"]).round(1)   # fração do grupo com texto repetido
repeated_per_cluster["category"] = repeated_per_cluster.index.map(name_of_each_cluster)                                                     # nome do grupo

repeated_per_cluster.sort_values("percentage", ascending=False)

,with_repeated_sentence,activities,percentage,category
cluster,,,,
15,43,50,86.0,Esporte
5,18,36,50.0,Música
4,8,26,30.8,Esporte
7,9,40,22.5,Artesanato
6,6,54,11.1,Sociedade e Debates
12,7,64,10.9,Artes Visuais
8,6,59,10.2,Esporte
1,4,54,7.4,Atividade Física e Bem-Estar
2,2,36,5.6,Cultura e Identidade


In [7]:
enrollment_sentence = activities_per_sentence.most_common(1)[0][0]                                                                                   # a frase mais repetida da base, que é do aviso de inscrição
has_enrollment_sentence = np.array([enrollment_sentence in sentences_of_the_activity for sentences_of_the_activity in sentences_of_each_activity])   # quais atividades têm essa frase

print("Frase:", enrollment_sentence[:90])
print("Em que grupos ela aparece:", pd.Series(cluster_of_each_activity[has_enrollment_sentence]).value_counts().to_dict())
print()
print("Títulos do grupo 15:")
for activity_title in activities.loc[cluster_of_each_activity == 15, "title"].head(8):
    print("   -", activity_title)

Frase: caso não sejam preenchidas nos primeiros cinco dias, elas serão abertas ao público em gera
Em que grupos ela aparece: {15: 20, 4: 2, 1: 2}

Títulos do grupo 15:
   - Juventudes ZN
   - Curumim
   - Esporte Adulto - Vôlei
   - Esporte para Pessoas Idosas - Multimodalidades
   - Esporte Jovem -  De 10 a 13 anos
   - Esporte Adulto - Handebol
   - Esporte Adulto - Iniciação ao Vôlei
   - Esporte Adulto - Futsal Masculino


**O que deu.** São 47 frases repetidas, em 107 das 693 atividades, todas do SESC, somando 6,6% das
palavras da base, o que é pouco texto na base como um todo.

Olhando por grupo, o retrato muda um pouco. O texto repetido se concentra no grupo 15, um dos três de
esporte, em que 43 das 50 atividades têm frase repetida e a frase do aviso de inscrição aparece em 20
delas. Pelos títulos, o grupo 15 é o dos cursos de esporte com inscrição ("Esporte Adulto - Vôlei",
"Esporte Adulto - Handebol"), então ali o assunto e o aviso andam juntos, e não conseguimos separar um
do outro só com esta contagem. O que sabemos é que esse grupo recebeu o nome "Esporte", igual aos outros
dois grupos de esporte, então na categoria o efeito some. Nos outros grupos a fração é baixa, e no de
Música (18 de 36) as frases repetidas são de outro tipo, como a indicação de idade e a apresentação do
professor.

**Decisão que isso sustenta.** Não limpamos o texto de inscrição, porque ele é pouco na base e, onde se
concentra, não mudou a categoria das atividades. O experimento 4 olha a mesma pergunta por outro lado.

## Experimento 2. Quem quase empata entre duas categorias?

A ideia mais simples para achar uma atividade com duas categorias é olhar a diferença entre a categoria
mais parecida e a segunda mais parecida: se as duas quase empatam, a atividade fica com as duas. Como
não sabíamos quantas atividades isso pegaria, testamos vários tamanhos de diferença.

In [8]:
# ====================================================================================================
#                         EXPERIMENTO 2: DIFERENÇA ENTRE A 1ª E A 2ª CATEGORIA
# ====================================================================================================
categories_from_closest_to_farthest = np.argsort(-similarity_to_categories, axis=1)   # posições das categorias, da mais parecida para a menos
row_numbers = np.arange(len(activities))                                              # 0, 1, 2... serve para pegar um valor por linha

position_of_first_category = categories_from_closest_to_farthest[:, 0]                   # categoria mais parecida de cada atividade
position_of_second_category = categories_from_closest_to_farthest[:, 1]                  # segunda mais parecida
first_similarity = similarity_to_categories[row_numbers, position_of_first_category]     # semelhança com a 1ª
second_similarity = similarity_to_categories[row_numbers, position_of_second_category]   # semelhança com a 2ª
difference_between_first_and_second = first_similarity - second_similarity               # pequena = quase empate

near_tie_counts = []                                                                                                                                  # uma linha por corte testado
for cutoff in [0.005, 0.01, 0.02, 0.03, 0.05]:
    number_of_near_ties = (difference_between_first_and_second <= cutoff).sum()                                                                       # atividades com diferença até o corte
    near_tie_counts.append({"cutoff": cutoff, "activities": number_of_near_ties, "percentage": round(100 * number_of_near_ties / len(activities))})   # guarda a contagem

pd.DataFrame(near_tie_counts)

,cutoff,activities,percentage
0,0.005,16,2
1,0.010,37,5
2,0.020,69,10
3,0.030,109,16
4,0.050,215,31


In [9]:
is_near_tie = difference_between_first_and_second <= 0.02   # corte que usamos nos próximos experimentos

first_category = np.array(category_names)[position_of_first_category]     # nome da 1ª categoria de cada atividade
second_category = np.array(category_names)[position_of_second_category]   # nome da 2ª categoria

pairs_of_near_ties = Counter()                                                                      # par de categorias -> quantas atividades quase empatam entre as duas
for activity_position in np.where(is_near_tie)[0]:
    pair = tuple(sorted([first_category[activity_position], second_category[activity_position]]))   # o par em ordem alfabética, para A+B e B+A contarem juntos
    pairs_of_near_ties[pair] += 1                                                                   # mais uma atividade nesse par

print("Atividades em que a 1ª categoria é a do grupo da Parte 2:", (first_category == category_of_each_activity).sum(), "de", len(activities))
print()
print("Pares mais comuns entre as quase empatadas (corte 0,02):")
for pair, number_of_activities in pairs_of_near_ties.most_common(8):
    print("  ", number_of_activities, "-", pair[0], "+", pair[1])

Atividades em que a 1ª categoria é a do grupo da Parte 2: 672 de 693

Pares mais comuns entre as quase empatadas (corte 0,02):
   7 - Atividade Física e Bem-Estar + Esporte
   7 - Artes Visuais + Artesanato
   6 - Esporte + Infância e Brincadeiras
   5 - Infância e Brincadeiras + Shows, Espetáculos e Performances
   4 - Infância e Brincadeiras + Literatura e Leitura
   4 - Cultura e Identidade + Shows, Espetáculos e Performances
   4 - Shows, Espetáculos e Performances + Sociedade e Debates
   4 - Cinema + Shows, Espetáculos e Performances


**O que deu.** Com corte de 0,02, são 69 atividades quase empatadas (10% da base), e com 0,05 já são 215
(31%). Os pares que mais aparecem fazem sentido para quem lê: Atividade Física com Esporte, Artes
Visuais com Artesanato, Infância com Shows.

Porém, achamos dois limites nessa regra. Ela nunca dá mais de 2 categorias, e só pega dupla em que as
duas categorias têm quase a mesma força, então se uma é bem mais forte que a outra, a segunda se perde
(aparece um caso assim no experimento 6).

**Decisão que isso sustenta.** A diferença entre a 1ª e a 2ª vira uma das regras candidatas do notebook
07, mas não a única.

## Experimento 3. As atividades vizinhas concordam com o empate?

O empate do experimento 2 sai da comparação com os centros, que são médias, e queríamos conferir por
outro caminho, olhando atividades de verdade. Para cada atividade pegamos as 10 mais parecidas com ela
(as vizinhas) e contamos quantas são da categoria que ficou em 2º lugar, já que, se o empate for real,
as quase empatadas devem ter mais vizinhas da 2ª categoria que as outras.

In [10]:
# ====================================================================================================
#                    EXPERIMENTO 3: AS 10 VIZINHAS MAIS PARECIDAS DE CADA ATIVIDADE
# ====================================================================================================
similarity_between_activities = normalized_embeddings @ normalized_embeddings.T   # semelhança de cada atividade com todas as outras
np.fill_diagonal(similarity_between_activities, -1)                               # a atividade não pode ser vizinha dela mesma

number_of_neighbors = 10                                                                          # quantas vizinhas olhamos
closest_neighbors = np.argsort(-similarity_between_activities, axis=1)[:, :number_of_neighbors]   # posições das 10 mais parecidas de cada atividade
categories_of_the_neighbors = category_of_each_activity[closest_neighbors]                        # a categoria de cada vizinha

neighbors_in_second_category = (categories_of_the_neighbors == second_category[:, None]).sum(axis=1)   # quantas das 10 são da 2ª categoria da atividade

print("Quase empatadas: mediana de", np.median(neighbors_in_second_category[is_near_tie]), "vizinhas da 2ª categoria |", round(100 * (neighbors_in_second_category[is_near_tie] >= 3).mean()), "% com 3 ou mais")
print("Resto da base:   mediana de", np.median(neighbors_in_second_category[~is_near_tie]), "vizinhas da 2ª categoria |", round(100 * (neighbors_in_second_category[~is_near_tie] >= 3).mean()), "% com 3 ou mais")

Quase empatadas: mediana de 3.0 vizinhas da 2ª categoria | 62 % com 3 ou mais
Resto da base:   mediana de 1.0 vizinhas da 2ª categoria | 22 % com 3 ou mais


**O que deu.** Entre as quase empatadas, a mediana é de 3 vizinhas da 2ª categoria, e 62% têm 3 ou mais,
enquanto no resto da base a mediana é 1 e só 22% têm 3 ou mais. Logo, os dois jeitos de olhar (os
centros e as vizinhas) apontam para as mesmas atividades.

**Decisão que isso sustenta.** As vizinhas servem de conferência para as regras do multirrótulo, e no
notebook 07 elas fazem o papel de juiz na hora de escolher a regra.

## Experimento 4. O empate vem do jeito de escrever?

Voltando ao receio do experimento 1: se o texto de inscrição ou o estilo do SESC estivessem causando os
empates, as quase empatadas teriam mais frase repetida e mais SESC que o resto da base.

In [11]:
# ====================================================================================================
#                 EXPERIMENTO 4: TEXTO REPETIDO E INSTITUIÇÃO ENTRE AS QUASE EMPATADAS
# ====================================================================================================
is_from_sesc = (activities["source"] == "SESC").values   # a atividade é do SESC?

print("Com frase repetida:  quase empatadas", round(100 * has_repeated_sentence[is_near_tie].mean()), "% | resto da base", round(100 * has_repeated_sentence[~is_near_tie].mean()), "%")
print("Do SESC:             quase empatadas", round(100 * is_from_sesc[is_near_tie].mean()), "% | resto da base", round(100 * is_from_sesc[~is_near_tie].mean()), "%")

Com frase repetida:  quase empatadas 4 % | resto da base 17 %
Do SESC:             quase empatadas 81 % | resto da base 81 %


**O que deu.** Das quase empatadas, só 4% têm frase repetida, contra 17% do resto, e o SESC é 81% nos
dois lados. Logo, os empates não vêm do texto burocrático nem da instituição.

**Decisão que isso sustenta.** Junto com o experimento 1, é o motivo de não limparmos o texto de
inscrição antes de rotular.

## Experimento 5. Uma amostra para ler

Número nenhum diz se as duplas de categoria fazem sentido para uma pessoa, então sorteamos 20 atividades
quase empatadas para ler uma por uma, e mais 5 em que a 1ª categoria ganha com folga (diferença de 0,15
ou mais), para servir de contraste.

In [12]:
# ====================================================================================================
#                    EXPERIMENTO 5: SORTEIO DE 20 QUASE EMPATADAS E 5 DE CONTRASTE
# ====================================================================================================
random_generator = np.random.default_rng(42)   # semente fixa: o sorteio é sempre o mesmo

near_tie_positions = np.where(difference_between_first_and_second <= 0.02)[0]       # de onde saem as 20
clear_winner_positions = np.where(difference_between_first_and_second >= 0.15)[0]   # de onde saem as 5 de contraste

sampled_near_ties = random_generator.choice(near_tie_positions, 20, replace=False)          # 20 quase empatadas, sem repetir
sampled_clear_winners = random_generator.choice(clear_winner_positions, 5, replace=False)   # 5 de contraste, com o mesmo gerador

sample_to_read = activities.loc[sampled_near_ties, ["title"]].copy()     # tabela com os títulos sorteados
sample_to_read["first_category"] = first_category[sampled_near_ties]     # o que a medida diz que a atividade é
sample_to_read["second_category"] = second_category[sampled_near_ties]   # a categoria que quase empatou
sample_to_read

,title,first_category,second_category
376,Dançando como nos filmes,"Shows, Espetáculos e Performances",Cinema
627,Você gosta de pedalar?,Sociedade e Debates,Atividade Física e Bem-Estar
121,PEQUENOS BRILHOS: A PASSARELA DA NOSSA INFÂNCIA,Reciclagem e Sustentabilidade,Infância e Brincadeiras
595,Espaço Trajetórias do Corpo,Esporte,Sociedade e Debates
590,"Corpo em Movimento, Corpo em História",Esporte,Sociedade e Debates
614,Fotografia na Lata,Artesanato,Artes Visuais
72,DIA DO SACI | JAXY JATERÊ – CONFECÇÃO DE MARCA...,Artesanato,Literatura e Leitura
30,OFICINA DE CONTAÇÃO DE HISTÓRIAS – COM SANDRA ...,Infância e Brincadeiras,Literatura e Leitura
331,Refugi Musical | África essencial,"Shows, Espetáculos e Performances",Cultura e Identidade
330,Práticas Aquáticas - Adulto,Esporte,Atividade Física e Bem-Estar


In [13]:
contrast_to_read = activities.loc[sampled_clear_winners, ["title"]].copy()     # as 5 em que a 1ª categoria ganha com folga
contrast_to_read["first_category"] = first_category[sampled_clear_winners]     # a categoria que ganhou
contrast_to_read["second_category"] = second_category[sampled_clear_winners]   # a segunda, bem mais longe
contrast_to_read

,title,first_category,second_category
307,Esporte Adulto - Handebol,Esporte,Infância e Brincadeiras
537,Cine Debate - Quando a arte imita a vida: suic...,Cinema,Sociedade e Debates
198,Vôlei para Pessoas Idosas,Esporte,Atividade Física e Bem-Estar
455,Recreação Monitorada de Badminton,Esporte,Atividade Física e Bem-Estar
402,Oriente-se,Atividade Física e Bem-Estar,Plantas e Alimentação


**O que deu.** Para ler a amostra precisamos primeiro combinar um critério: a atividade recebe a
categoria X se quem procura X provavelmente se interessaria por ela. A ordem também importa, já que a 1ª
categoria é o que a atividade é e as seguintes são interesses relacionados.

Com esse critério, várias duplas estão certas, por exemplo "Mundo Mágico: festa para as crianças" (Shows
e Infância), "Dançando como nos filmes" (Shows em 1º, Cinema em 2º) e "Corpo em Movimento, Corpo em
História" (Esporte e Sociedade e Debates). Duas estão erradas: "Conexão Jovem", que é empréstimo de
jogos de tabuleiro para jovens, aparece com Esporte, e "Pequenos Brilhos", um desfile infantil que não
fala de reciclagem, aparece com Reciclagem. Em "Você gosta de pedalar?" não deu para julgar.

**Decisão que isso sustenta.** O critério de leitura virou a regra que o grupo usa na conferência humana
da Parte 3 (notebook 08). Além disso, a amostra mostrou que a medida erra com confiança em alguns casos,
então a qualidade tem que ser conferida por pessoas, e não só por conta.

## Experimento 6. A partir de que semelhança uma categoria "vale"?

Outra regra possível é dar à atividade toda categoria com que ela seja parecida o bastante, e para isso
precisávamos saber o que é "o bastante". Olhamos, em cada categoria, a semelhança das atividades que são
dela (a mediana e a beirada, que é o valor que 90% delas ultrapassam) e comparamos com a semelhança que
uma atividade costuma ter com as categorias que não são a dela.

In [14]:
# ====================================================================================================
#                EXPERIMENTO 6: A ESCALA DAS SEMELHANÇAS E A BEIRADA DE CADA CATEGORIA
# ====================================================================================================
position_of_own_category = np.array([category_names.index(category_name) for category_name in category_of_each_activity])   # coluna da categoria de cada atividade
similarity_to_own_category = similarity_to_categories[row_numbers, position_of_own_category]                                # semelhança com a própria categoria

is_own_category = np.zeros(similarity_to_categories.shape, dtype=bool)        # tabela de verdadeiro e falso, do tamanho da de semelhanças
is_own_category[row_numbers, position_of_own_category] = True                 # marca a coluna da própria categoria em cada linha
similarity_to_other_categories = similarity_to_categories[~is_own_category]   # todas as semelhanças com categoria alheia

print("Com categoria que não é a dela: mediana", round(np.median(similarity_to_other_categories), 2), "| 90% ficam abaixo de", round(np.percentile(similarity_to_other_categories, 90), 2))

scale_per_category = []                                                                                                                                                                      # uma linha por categoria
for category_name in category_names:
    similarities_of_the_members = similarity_to_own_category[category_of_each_activity == category_name]                                                                                     # semelhança dos membros com o próprio centro
    scale_per_category.append({"category": category_name, "median": round(np.median(similarities_of_the_members), 2), "border": round(np.percentile(similarities_of_the_members, 10), 2)})   # mediana e beirada (percentil 10)

scale_per_category = pd.DataFrame(scale_per_category)   # lista vira tabela
scale_per_category

Com categoria que não é a dela: mediana 0.51 | 90% ficam abaixo de 0.62


,category,median,border
0,Artes Visuais,0.74,0.65
1,Artesanato,0.75,0.63
2,Atividade Física e Bem-Estar,0.73,0.66
3,Cinema,0.65,0.54
4,Cultura e Identidade,0.73,0.67
5,Esporte,0.74,0.58
6,Infância e Brincadeiras,0.76,0.68
7,Literatura e Leitura,0.72,0.61
8,Música,0.79,0.70
9,Plantas e Alimentação,0.70,0.58


In [15]:
# ----------------------------------------------------------------------------------------------------
#                     E SE A ATIVIDADE RECEBESSE TODA CATEGORIA ACIMA DA BEIRADA?
# ----------------------------------------------------------------------------------------------------
border_of_each_category = np.array([np.percentile(similarity_to_own_category[category_of_each_activity == category_name], 10) for category_name in category_names])   # as 13 beiradas, sem arredondar
is_above_border = similarity_to_categories >= border_of_each_category                                                                                                 # a atividade passa da beirada da categoria?
labels_by_border = is_above_border.sum(axis=1)                                                                                                                        # quantas categorias cada atividade receberia

print("Sem nenhuma categoria:", (labels_by_border == 0).sum())
print("Com 1:", (labels_by_border == 1).sum(), "| com 2:", (labels_by_border == 2).sum(), "| com 3:", (labels_by_border == 3).sum(), "| com 4 ou mais:", (labels_by_border >= 4).sum())

Sem nenhuma categoria: 59
Com 1: 272 | com 2: 146 | com 3: 71 | com 4 ou mais: 145


In [16]:
# ----------------------------------------------------------------------------------------------------
#                                      DOIS CASOS LIDOS DE PERTO
# ----------------------------------------------------------------------------------------------------
def show_values_of_activity(piece_of_title, values_per_category):
    for activity_position in np.where(activities["title"].str.contains(piece_of_title, regex=False))[0]:
        values_of_the_activity = pd.Series(values_per_category[activity_position], index=category_names).sort_values(ascending=False)   # os 13 valores, do maior para o menor
        print(activities.loc[activity_position, "title"][:80])
        print("   ", ", ".join(category_name + " " + str(round(value, 2)) for category_name, value in values_of_the_activity.head(4).items()), "| menor:", round(values_of_the_activity.min(), 2))

show_values_of_activity("Artesania da Produção Cultural", similarity_to_categories)
show_values_of_activity("Cine Debate - Quando a arte imita a vida", similarity_to_categories)

Artesania da Produção Cultural - Como executar um projeto cultural
    Artes Visuais 0.65, Artesanato 0.63, Cultura e Identidade 0.59, Reciclagem e Sustentabilidade 0.57 | menor: 0.49
Cine Debate - Quando a arte imita a vida: suicídio em cena - Alarme silencioso
    Cinema 0.7, Sociedade e Debates 0.53, Shows, Espetáculos e Performances 0.48, Literatura e Leitura 0.45 | menor: 0.31


In [17]:
# ----------------------------------------------------------------------------------------------------
#                                 ATIVIDADES COM A DESCRIÇÃO QUEBRADA
# ----------------------------------------------------------------------------------------------------
has_broken_description = (activities["description"].str.strip() == "Descrição não encontrada").values   # a coleta não achou a descrição

broken_activities = activities.loc[has_broken_description, ["title", "split", "category"]].copy()                 # quais são e onde caíram
broken_activities["highest_similarity"] = similarity_to_categories[has_broken_description].max(axis=1).round(2)   # a maior semelhança de cada uma
broken_activities

,title,split,category,highest_similarity
405,Tênis de mesa,train,Esporte,0.32
595,Espaço Trajetórias do Corpo,train,Cinema,0.35
675,Batuco,train,Cinema,0.27


**O que deu.** Uma atividade tem semelhança mediana de 0,51 com as categorias que não são a dela, e 90%
dessas semelhanças ficam abaixo de 0,62. Com a própria categoria, a mediana vai de 0,64 (Shows) a 0,79
(Música), e as beiradas vão de 0,54 (Cinema) a 0,70 (Música).

Dar toda categoria acima da beirada é generoso demais: 59 atividades ficam sem nenhuma, 272 com uma, 146
com duas, 71 com três e 145 com quatro ou mais.

Os dois casos lidos mostram limites diferentes. "Artesania da Produção Cultural" é uma oficina de gestão
de projeto cultural, e as duas maiores semelhanças dela, Artes Visuais 0,65 e Artesanato 0,63, estão em
cima da beirada de cada categoria, com a linha toda plana (de 0,49 a 0,65), então ela não se encaixa em
nada. O "Cine Debate", por outro lado, fica com Cinema 0,70 e Sociedade e Debates 0,53, abaixo da
beirada de Debates, embora a descrição fale quase só do debate sobre saúde mental, o que faz dele uma
dupla que a medida perde.

Foi aqui também que achamos um erro de dados: 3 atividades entraram na base com a descrição "Descrição
não encontrada", por falha da coleta. As 3 estão no treino, duas no grupo de Cinema e uma em Esporte.

**Decisões que isso sustenta.** A beirada sozinha não serve de regra, mas entra como candidata no
notebook 07 e volta no aviso de revisão humana (uma atividade abaixo da beirada da própria categoria não
se encaixa bem em nada). As 3 atividades de descrição quebrada continuam na base, porque são poucas,
estão no treino e tirá-las pediria mais uma justificativa para um efeito pequeno. Alguns dos próximos
experimentos foram feitos sem elas, e avisamos quando for o caso.

## Experimento 7. Dividir a atividade entre as categorias

As duas regras anteriores têm um defeito parecido, já que a da diferença só olha as duas primeiras
categorias e a da beirada olha cada categoria sozinha. Pensamos então numa terceira, que olha as 13 de
uma vez, com a imagem de uma pizza dividida entre as categorias: quanto mais parecida a categoria é com
a atividade, maior a fatia dela, e as fatias somam 100%. A atividade recebe toda categoria que ficar com
uma fatia mínima.

A conta que faz essa divisão se chama softmax, e ela tem um botão, a temperatura (T): com T baixo, a
categoria mais parecida leva quase a pizza inteira, e com T alto as fatias ficam quase do mesmo tamanho.
Como não sabíamos que T usar, montamos a tabela com vários. As contagens abaixo deixam de fora as 3
atividades de descrição quebrada.

In [18]:
# ====================================================================================================
#                    EXPERIMENTO 7: FATIAS POR CATEGORIA (SOFTMAX COM TEMPERATURA)
# ====================================================================================================
def softmax_with_temperature(similarities, temperature):
    scaled_similarities = similarities / temperature                                             # temperatura baixa aumenta a distância entre as categorias
    scaled_similarities = scaled_similarities - scaled_similarities.max(axis=1, keepdims=True)   # tira o maior valor de cada linha, para a exponencial não estourar
    exponentials = np.exp(scaled_similarities)                                                   # a exponencial deixa tudo positivo e destaca os maiores
    return exponentials / exponentials.sum(axis=1, keepdims=True)                                # divide pela soma da linha: as fatias somam 1

def count_labels_per_setting(similarities):
    counts_per_setting = []                                                                                                                                                                                                                                                                                                                # uma linha por combinação de temperatura e fatia mínima
    for temperature in [0.01, 0.02, 0.03, 0.05]:
        for minimum_share in [0.20, 0.25]:
            shares = softmax_with_temperature(similarities, temperature)[~has_broken_description]                                                                                                                                                                                                                                          # fatias, sem as 3 de descrição quebrada
            number_of_labels = (shares >= minimum_share).sum(axis=1)                                                                                                                                                                                                                                                                       # categorias com fatia suficiente em cada atividade
            largest_share = shares.max(axis=1)                                                                                                                                                                                                                                                                                             # a maior fatia de cada atividade
            counts_per_setting.append({"temperature": temperature, "minimum_share": minimum_share, "1_label": (number_of_labels == 1).sum(), "2_labels": (number_of_labels == 2).sum(), "3_labels": (number_of_labels == 3).sum(), "4_or_more": (number_of_labels >= 4).sum(), "largest_share_below_40%": (largest_share < 0.40).sum()})   # guarda as contagens
    return pd.DataFrame(counts_per_setting)

count_labels_per_setting(similarity_to_categories)

,temperature,minimum_share,1_label,2_labels,3_labels,4_or_more,largest_share_below_40%
0,0.01,0.20,640,49,1,0,1
1,0.01,0.25,652,38,0,0,1
2,0.02,0.20,603,79,8,0,6
3,0.02,0.25,627,61,1,0,6
4,0.03,0.20,565,115,9,0,25
5,0.03,0.25,614,73,1,0,25
6,0.05,0.20,545,132,10,0,123
7,0.05,0.25,613,67,0,0,123


In [19]:
# ----------------------------------------------------------------------------------------------------
#                         AS MESMAS ATIVIDADES DO EXPERIMENTO 5, COM T = 0,02
# ----------------------------------------------------------------------------------------------------
shares_per_category = softmax_with_temperature(similarity_to_categories, 0.02)   # fatias com a temperatura do meio da tabela

for piece_of_title in ["Dançando como nos filmes", "Conexão Jovem", "PEQUENOS BRILHOS", "Artesania da Produção Cultural", "Espaço Trajetórias do Corpo"]:
    show_values_of_activity(piece_of_title, shares_per_category)

Dançando como nos filmes
    Shows, Espetáculos e Performances 0.45, Cinema 0.33, Atividade Física e Bem-Estar 0.2, Esporte 0.01 | menor: 0.0
Conexão Jovem
    Infância e Brincadeiras 0.46, Esporte 0.43, Literatura e Leitura 0.08, Reciclagem e Sustentabilidade 0.02 | menor: 0.0
PEQUENOS BRILHOS: A PASSARELA DA NOSSA INFÂNCIA
    Reciclagem e Sustentabilidade 0.49, Infância e Brincadeiras 0.3, Shows, Espetáculos e Performances 0.2, Literatura e Leitura 0.0 | menor: 0.0
Artesania da Produção Cultural - Como executar um projeto cultural
    Artes Visuais 0.58, Artesanato 0.32, Cultura e Identidade 0.03, Reciclagem e Sustentabilidade 0.02 | menor: 0.0
Espaço Trajetórias do Corpo
    Atividade Física e Bem-Estar 0.6, Infância e Brincadeiras 0.29, Esporte 0.09, Artes Visuais 0.02 | menor: 0.0
Espaço Trajetórias do Corpo
    Esporte 0.29, Sociedade e Debates 0.21, Cinema 0.16, Artes Visuais 0.08 | menor: 0.0


**O que deu.** A quantidade de categorias aparece sozinha, sem a gente fixar que são duas. Com T = 0,02
e fatia mínima de 20%, 603 atividades ficam com uma categoria, 79 com duas e 8 com três. "Dançando como
nos filmes" fica com Shows 45%, Cinema 33% e Atividade Física 20%, e a terceira está certa, porque a
descrição diz que são aulas de ritmo e coordenação.

A maior fatia também serve de alerta. Existem duas atividades chamadas "Espaço Trajetórias do Corpo",
uma com a descrição de verdade, que fica com Atividade Física 60%, e a de descrição quebrada, que fica
com a maior fatia em 29%, sem nenhuma categoria se destacar.

Porém, dois problemas continuam. Os erros confiantes do experimento 5 seguem errados: Conexão Jovem
(Infância 46%, Esporte 43%) e Pequenos Brilhos (Reciclagem 49%, Infância 30%). Além disso, "Artesania",
que não se encaixa em nada, sai com Artes Visuais 58%, porque as fatias só comparam as categorias entre
si, então uma linha plana com um valor um pouco maior parece uma resposta firme. Para pegar esse caso é
preciso um sinal que olhe a semelhança em si, como a beirada do experimento 6.

Ficou em aberto como escolher T e a fatia mínima sem ser no olho.

### A mesma conta com os 16 centros dos grupos

Até aqui cada categoria tinha um centro só, mas Esporte são três grupos e Shows são dois, e a média de
três grupos diferentes pode cair num lugar que não representa nenhum deles. Por isso refizemos a conta
com um centro por grupo (16), e a semelhança da atividade com uma categoria passa a ser a maior entre os
grupos daquela categoria.

In [20]:
# ----------------------------------------------------------------------------------------------------
#              EXPERIMENTO 7 COM UM CENTRO POR GRUPO (16) EM VEZ DE UM POR CATEGORIA (13)
# ----------------------------------------------------------------------------------------------------
number_of_clusters = 16   # os grupos da Parte 2

cluster_centers = []                                                                                # um centro por grupo
for cluster_number in range(number_of_clusters):
    embeddings_of_the_cluster = normalized_embeddings[cluster_of_each_activity == cluster_number]   # vetores das atividades do grupo
    cluster_centers.append(embeddings_of_the_cluster.mean(axis=0))                                  # o centro é a média deles
cluster_centers = np.array(cluster_centers)                                                         # lista vira matriz de 16 linhas
cluster_centers = cluster_centers / np.linalg.norm(cluster_centers, axis=1, keepdims=True)          # centros com comprimento 1

similarity_to_clusters = normalized_embeddings @ cluster_centers.T   # uma coluna por grupo

similarity_to_categories_by_clusters = []                                                                                                                  # uma coluna por categoria, montada a partir dos grupos
for category_name in category_names:
    clusters_of_the_category = [cluster_number for cluster_number in range(number_of_clusters) if name_of_each_cluster[cluster_number] == category_name]   # os grupos que receberam esse nome
    similarity_to_categories_by_clusters.append(similarity_to_clusters[:, clusters_of_the_category].max(axis=1))                                           # vale o grupo mais parecido
similarity_to_categories_by_clusters = np.array(similarity_to_categories_by_clusters).T                                                                    # de volta ao formato atividade x categoria

first_category_by_clusters = np.array(category_names)[similarity_to_categories_by_clusters.argmax(axis=1)]   # a categoria mais parecida, agora pelos 16 centros

print("1ª categoria igual à do grupo da Parte 2, com 13 centros:", (first_category == category_of_each_activity).sum(), "de", len(activities))
print("1ª categoria igual à do grupo da Parte 2, com 16 centros:", (first_category_by_clusters == category_of_each_activity).sum(), "de", len(activities))
count_labels_per_setting(similarity_to_categories_by_clusters)

1ª categoria igual à do grupo da Parte 2, com 13 centros: 672 de 693
1ª categoria igual à do grupo da Parte 2, com 16 centros: 692 de 693


,temperature,minimum_share,1_label,2_labels,3_labels,4_or_more,largest_share_below_40%
0,0.01,0.20,668,22,0,0,0
1,0.01,0.25,673,17,0,0,0
2,0.02,0.20,631,59,0,0,0
3,0.02,0.25,651,39,0,0,0
4,0.03,0.20,597,92,1,0,9
5,0.03,0.25,638,52,0,0,9
6,0.05,0.20,564,124,2,0,93
7,0.05,0.25,633,54,0,0,93


**O que deu.** Com 13 centros, a categoria mais parecida coincide com a do grupo da Parte 2 em 672 das
693 atividades, e com 16 centros, em 692, então os 16 centros repetem a resposta de categoria única que
já tínhamos, e os 13 não. Com 16 centros as fatias também ficam mais decididas: com T = 0,02 e 20%, são
631 atividades com uma categoria e 59 com duas.

**Decisões que isso sustenta.** A regra das fatias entra como terceira candidata, e o multirrótulo passa
a ser medido pelos 16 centros dos grupos, valendo o grupo mais parecido de cada categoria. Além disso, T
e a fatia mínima passam a ser escolhidos por um juiz, as vizinhas do experimento 3, só com o treino
(notebook 07).

## Experimento 8. Existem atividades que não são de grupo nenhum?

Se muitas atividades estivessem longe do próprio grupo, e parecidas entre si, poderia haver assuntos que
os 16 grupos não pegaram. Chamamos de órfã a atividade que não é mais parecida com o próprio grupo do
que uma atividade costuma ser de um grupo alheio, e depois juntamos as órfãs entre si para ver se formam
algum assunto novo. Este experimento foi feito sem as 3 atividades de descrição quebrada.

In [21]:
# ====================================================================================================
#                           EXPERIMENTO 8: ATIVIDADES LONGE DO PRÓPRIO GRUPO
# ====================================================================================================
valid_embeddings = normalized_embeddings[~has_broken_description]              # vetores das 690 atividades com descrição
valid_clusters = cluster_of_each_activity[~has_broken_description]             # grupo de cada uma
valid_titles = activities.loc[~has_broken_description, "title"].to_numpy()     # título de cada uma
valid_sources = activities.loc[~has_broken_description, "source"].to_numpy()   # instituição de cada uma
valid_row_numbers = np.arange(len(valid_embeddings))                           # 0, 1, 2... para pegar um valor por linha

valid_cluster_centers = []                                                                                     # centros dos 16 grupos, agora sem as 3 quebradas
for cluster_number in range(number_of_clusters):
    valid_cluster_centers.append(valid_embeddings[valid_clusters == cluster_number].mean(axis=0))              # média dos vetores do grupo
valid_cluster_centers = np.array(valid_cluster_centers)                                                        # lista vira matriz
valid_cluster_centers = valid_cluster_centers / np.linalg.norm(valid_cluster_centers, axis=1, keepdims=True)   # centros com comprimento 1

valid_similarity_to_clusters = valid_embeddings @ valid_cluster_centers.T                     # semelhança de cada atividade com cada grupo
similarity_to_own_cluster = valid_similarity_to_clusters[valid_row_numbers, valid_clusters]   # com o próprio grupo

is_own_cluster = np.zeros(valid_similarity_to_clusters.shape, dtype=bool)      # tabela de verdadeiro e falso
is_own_cluster[valid_row_numbers, valid_clusters] = True                       # marca o próprio grupo em cada linha
similarity_to_other_clusters = valid_similarity_to_clusters[~is_own_cluster]   # todas as semelhanças com grupo alheio

orphan_limit = np.percentile(similarity_to_other_clusters, 90)   # 90% das semelhanças com grupo alheio ficam abaixo disso
is_orphan = similarity_to_own_cluster < orphan_limit             # não é mais parecida com o próprio grupo do que se costuma ser de um alheio

print("Com o próprio grupo: mediana", round(np.median(similarity_to_own_cluster), 2))
print("Com grupo alheio: mediana", round(np.median(similarity_to_other_clusters), 2), "| 90% abaixo de", round(orphan_limit, 2))
print("Órfãs:", is_orphan.sum(), "de", len(valid_embeddings), "(", round(100 * is_orphan.mean()), "% )")
print("De que instituição:", pd.Series(valid_sources[is_orphan]).value_counts().to_dict())

Com o próprio grupo: mediana 0.72
Com grupo alheio: mediana 0.5 | 90% abaixo de 0.62
Órfãs: 90 de 690 ( 13 % )
De que instituição: {'SESC': 80, 'Fábricas de Cultura': 10}


In [22]:
similarity_between_valid_activities = valid_embeddings @ valid_embeddings.T                                # semelhança entre todas as atividades com descrição
positions_above_diagonal = np.triu_indices(len(valid_embeddings), 1)                                       # cada par de atividades uma vez só
similarity_of_each_pair = similarity_between_valid_activities[positions_above_diagonal]                    # semelhança de cada par
pair_is_in_same_cluster = (valid_clusters[:, None] == valid_clusters[None, :])[positions_above_diagonal]   # as duas do par são do mesmo grupo?

print("Entre duas atividades quaisquer: mediana", round(np.median(similarity_of_each_pair), 2), "| 90% abaixo de", round(np.percentile(similarity_of_each_pair, 90), 2), "| 99% abaixo de", round(np.percentile(similarity_of_each_pair, 99), 2))
print("Entre duas do mesmo grupo: mediana", round(np.median(similarity_of_each_pair[pair_is_in_same_cluster]), 2))

Entre duas atividades quaisquer: mediana

 0.37 | 90% abaixo de 0.51 | 99% abaixo de 0.66
Entre duas do mesmo grupo: mediana 0.5


In [23]:
# ----------------------------------------------------------------------------------------------------
#                                     EM QUE GRUPOS ESTÃO AS ÓRFÃS
# ----------------------------------------------------------------------------------------------------
orphans_per_cluster = pd.DataFrame({"cluster": valid_clusters, "is_orphan": is_orphan}).groupby("cluster")["is_orphan"].agg(["sum", "count"])   # órfãs e total de cada grupo
orphans_per_cluster.columns = ["orphans", "activities"]                                                                                         # nomes mais claros para as duas contagens
orphans_per_cluster["category"] = orphans_per_cluster.index.map(name_of_each_cluster)                                                           # nome do grupo

orphans_per_cluster.sort_values("orphans", ascending=False).head(8)

,orphans,activities,category
cluster,,,
10,19,54,Cinema
6,13,54,Sociedade e Debates
3,12,47,"Shows, Espetáculos e Performances"
14,10,40,"Shows, Espetáculos e Performances"
4,6,26,Esporte
8,6,58,Esporte
13,5,17,Plantas e Alimentação
0,5,45,Literatura e Leitura


In [24]:
# ----------------------------------------------------------------------------------------------------
#                           OS ERROS CONFIANTES DO EXPERIMENTO 5 SÃO ÓRFÃS?
# ----------------------------------------------------------------------------------------------------
for piece_of_title in ["Conexão Jovem", "Artesania da Produção Cultural", "Cine Debate - Quando a arte imita a vida", "PEQUENOS BRILHOS", "Você gosta de pedalar?"]:
    for activity_position in np.where(pd.Series(valid_titles).str.contains(piece_of_title, regex=False))[0]:
        print(round(similarity_to_own_cluster[activity_position], 2), "com o próprio grupo | órfã:", is_orphan[activity_position], "|", valid_titles[activity_position][:60])

0.65 com o próprio grupo | órfã: False | Conexão Jovem
0.65 com o próprio grupo | órfã: False | Artesania da Produção Cultural - Como executar um projeto cu
0.7 com o próprio grupo | órfã: False | Cine Debate - Quando a arte imita a vida: suicídio em cena -
0.67 com o próprio grupo | órfã: False | PEQUENOS BRILHOS: A PASSARELA DA NOSSA INFÂNCIA
0.53 com o próprio grupo | órfã: True | Você gosta de pedalar?


In [25]:
# ----------------------------------------------------------------------------------------------------
#                                     AS ÓRFÃS AGRUPADAS ENTRE SI
# ----------------------------------------------------------------------------------------------------
orphan_embeddings = valid_embeddings[is_orphan]   # vetores só das órfãs
orphan_titles = valid_titles[is_orphan]           # títulos só das órfãs

for distance_limit in [0.55, 0.60, 0.65]:
    orphan_clustering = AgglomerativeClustering(n_clusters=None, metric="cosine", linkage="average", distance_threshold=distance_limit)   # junta órfãs enquanto a distância média entre elas for menor que o limite
    orphan_groups = orphan_clustering.fit_predict(orphan_embeddings)                                                                      # em que grupo de órfãs cada uma caiu
    sizes_of_orphan_groups = pd.Series(orphan_groups).value_counts()                                                                      # tamanho de cada grupo de órfãs
    groups_with_three_or_more = sizes_of_orphan_groups[sizes_of_orphan_groups >= 3]                                                       # só os que dão para ler
    print("Limite", distance_limit, "| grupos com 3 órfãs ou mais:", len(groups_with_three_or_more), "| órfãs neles:", groups_with_three_or_more.sum())

Limite 0.55 | grupos com 3 órfãs ou mais: 5 | órfãs neles: 16
Limite 0.6 | grupos com 3 órfãs ou mais: 11 | órfãs neles: 49
Limite 0.65 | grupos com 3 órfãs ou mais: 15 | órfãs neles: 76


In [26]:
orphan_clustering = AgglomerativeClustering(n_clusters=None, metric="cosine", linkage="average", distance_threshold=0.60)   # o limite do meio, para ler os grupos
orphan_groups = orphan_clustering.fit_predict(orphan_embeddings)                                                            # em que grupo de órfãs cada uma caiu
sizes_of_orphan_groups = pd.Series(orphan_groups).value_counts()                                                            # tamanho de cada grupo de órfãs

for orphan_group, group_size in sizes_of_orphan_groups[sizes_of_orphan_groups >= 3].items():
    print(group_size, "órfãs:", " | ".join(activity_title[:38] for activity_title in orphan_titles[orphan_groups == orphan_group][:6]))

7 órfãs: BALÉ CASA DE CULTURA SÃO LUÍS APRESENT | Curumim | Curso de Atuações | Dançando como nos filmes | Dança de Rua: Ela Dança Eu Danço | Circuito de Dublê de Ação
6 órfãs: Sophia Chablau e Uma Enorme Perda de T | Em Primeira Pessoa: Hyldon | Romulo Fróes, "Cão": 20 anos | Tributo a Cristina Buarque e Mauro Dua | Kiko Dinucci | O Barco, de Grada Kilomba
5 órfãs: BIBLIOCINE: UMA FAMÍLIA DE ARREPIAR –  | Cine-concerto "frozen " | A Praga | A Sombra do Pai | MONSTRO - tenho sonhado com homens
5 órfãs: Mostra Tecnologias de Todos os Tempos  | Digital de Bolso: Verdade em disputa n | Plantão de dúvidas: Bê-a-bá digital | Bê-a-Bá Digital: Módulo II | Fraudes e Golpes na Era da Inteligênci
5 órfãs: Luta olímpica com Aline Silva | Festival de futsal para mulheres | Luta olímpica | Bumerangue | PSE - Aula aberta de Jogos de Lutas
4 órfãs: PICADEIRO DA BRASA: O CIRCO CHEGOU! | FÁBRICA CAPÃO 14 ANOS – INTERVENÇÃO CI | Fuleragem | Encontro de Gigantes
4 órfãs: 6° EXTREMUM FEST | Videoclipes Qu

**O que deu.** São 90 órfãs em 690 (13%), na mesma proporção de SESC e Fábricas que a base, e elas se
concentram nos grupos de formato: o de Cinema (19 de 54), o de Sociedade e Debates (13 de 54) e os dois
de Shows (12 de 47 e 10 de 40).

Os erros confiantes do experimento 5 não são órfãs, já que Conexão Jovem, Artesania, Cine Debate e
Pequenos Brilhos têm entre 0,65 e 0,70 de semelhança com o próprio grupo, como um membro comum, e só
"Você gosta de pedalar?" (0,53) é órfã. Logo, medir a distância até o grupo não acha esses erros.

Agrupadas entre si, as órfãs formam quase só assuntos que já existem (filmes, shows de música, rock,
circo, lutas, dança), e os candidatos a assunto novo são fracos, com 3 a 5 atividades cada, como
tecnologia e sociedade (desinformação, golpes com inteligência artificial).

Lendo os grupos de filmes e de shows, notamos o motivo de Cinema e Shows concentrarem as órfãs: filme e
show são descritos pela sinopse, então o embedding aproxima a atividade do assunto da história, e não do
formato. Essa é a mesma mistura de formato com assunto que apontamos nas categorias dos sites, no
notebook 01.

**Decisão que isso sustenta.** Não criamos categorias novas na Parte 3, então as 13 continuam, e "mais
de uma categoria" quer dizer que a atividade pode receber mais de uma delas. Descobrir categorias novas
a partir das atividades que não se encaixam fica como trabalho futuro.

## Experimento 9. O que acontece se refizermos os grupos só com o treino?

Para o teste valer como "atividades novas", os grupos precisam ser formados só com o treino. A primeira
ideia foi a mais direta, rodar o mesmo K-Means, com a mesma semente 42, só nas atividades de treino, e
encaixar o teste depois, já que achávamos que a mesma semente daria os mesmos grupos. Este experimento
também foi feito sem as 3 de descrição quebrada.

In [27]:
# ====================================================================================================
#                  EXPERIMENTO 9: O MESMO K-MEANS, COM A MESMA SEMENTE, SÓ NO TREINO
# ====================================================================================================
is_train = (activities["split"] == "train").values    # a atividade é do treino?
is_test = ~is_train                                   # ou do teste
is_valid_train = is_train & ~has_broken_description   # treino sem as 3 de descrição quebrada

train_embeddings = activity_embeddings[is_valid_train]   # vetores sem normalizar, como no notebook 05
test_embeddings = activity_embeddings[is_test]           # o teste só entra depois

kmeans_on_train = KMeans(n_clusters=number_of_clusters, random_state=42, n_init=10)   # igual ao do notebook 05
new_clusters_of_train = kmeans_on_train.fit_predict(train_embeddings)                 # grupos formados só com o treino
new_clusters_of_test = kmeans_on_train.predict(test_embeddings)                       # cada atividade de teste vai para o centro mais perto

old_clusters_of_train = cluster_of_each_activity[is_valid_train]   # grupo da Parte 2 das mesmas atividades
old_clusters_of_test = cluster_of_each_activity[is_test]           # idem para o teste

members_shared = pd.crosstab(new_clusters_of_train, old_clusters_of_train)             # linha: grupo novo; coluna: grupo antigo; valor: atividades em comum
old_cluster_of_each_new = members_shared.idxmax(axis=1)                                # o grupo antigo com que cada grupo novo mais divide membros
name_of_each_new_cluster = old_cluster_of_each_new.map(name_of_each_cluster)           # o grupo novo herda o nome desse grupo antigo
purity_of_each_new_cluster = members_shared.max(axis=1) / members_shared.sum(axis=1)   # que fração do grupo novo veio desse grupo antigo

new_category_of_train = name_of_each_new_cluster[new_clusters_of_train].to_numpy()   # categoria de cada atividade de treino no agrupamento novo
new_category_of_test = name_of_each_new_cluster[new_clusters_of_test].to_numpy()     # idem para o teste
kept_in_train = new_category_of_train == category_of_each_activity[is_valid_train]   # ficou na mesma categoria da Parte 2?
kept_in_test = new_category_of_test == category_of_each_activity[is_test]            # idem para o teste

print("Mantêm a categoria: treino", round(100 * kept_in_train.mean(), 1), "% | teste", round(100 * kept_in_test.mean(), 1), "% | total", kept_in_train.sum() + kept_in_test.sum(), "de", len(kept_in_train) + len(kept_in_test))
print("Rand ajustado entre o agrupamento antigo e o novo:", round(adjusted_rand_score(np.concatenate([old_clusters_of_train, old_clusters_of_test]), np.concatenate([new_clusters_of_train, new_clusters_of_test])), 2))
print("Grupos antigos que não deram nome a nenhum grupo novo:", sorted(set(range(number_of_clusters)) - set(old_cluster_of_each_new)))
print("Os grupos novos mais misturados (fração que veio de um mesmo grupo antigo):", purity_of_each_new_cluster.sort_values().round(2).head(4).tolist())

Mantêm a categoria: treino 73.9 % | teste 71.2 % | total 506 de 690
Rand ajustado entre o agrupamento antigo e o novo: 0.46
Grupos antigos que não deram nome a nenhum grupo novo: [4, 8, 14]
Os grupos novos mais misturados (fração que veio de um mesmo grupo antigo): [0.27, 0.33, 0.36, 0.41]


In [28]:
old_and_new_categories = pd.DataFrame({"old_category": np.concatenate([category_of_each_activity[is_valid_train], category_of_each_activity[is_test]]), "new_category": np.concatenate([new_category_of_train, new_category_of_test])})   # categoria antiga e nova de cada atividade
changed_category = old_and_new_categories[old_and_new_categories["old_category"] != old_and_new_categories["new_category"]]                                                                                                               # só as que mudaram

changed_category.value_counts().head(3)

old_category                       new_category                
Esporte                            Atividade Física e Bem-Estar    30
Shows, Espetáculos e Performances  Cultura e Identidade             9
                                   Música                           9
Name: count, dtype: int64

Das 690 atividades, só 506 (73%) ficaram na mesma categoria, e três grupos antigos sumiram (dois dos
três de Esporte e um dos dois de Shows), com a troca mais comum indo de Esporte para Atividade Física e
Bem-Estar, em 30 atividades. Como não esperávamos isso, antes de culpar a retirada do teste fizemos um
controle: manter exatamente as mesmas 693 atividades e trocar só a semente.

In [29]:
# ----------------------------------------------------------------------------------------------------
#                        CONTROLE: AS MESMAS ATIVIDADES, TROCANDO SÓ A SEMENTE
# ----------------------------------------------------------------------------------------------------
seeds_to_compare = range(10)   # sementes diferentes da 42

clusters_per_seed = {}                                                                                                               # semente -> agrupamento das 693 atividades
for seed in seeds_to_compare:
    clusters_per_seed[seed] = KMeans(n_clusters=number_of_clusters, random_state=seed, n_init=10).fit_predict(activity_embeddings)   # o K-Means do notebook 05, mudando só a semente

agreement_with_original = [adjusted_rand_score(cluster_of_each_activity, clusters_per_seed[seed]) for seed in seeds_to_compare]   # o quanto cada semente agrupa igual à 42

print("Mesmas 693 atividades, outra semente, contra o agrupamento da Parte 2:")
print("   Rand ajustado de", round(min(agreement_with_original), 2), "a", round(max(agreement_with_original), 2), "| média", round(np.mean(agreement_with_original), 2))

Mesmas 693 atividades, outra semente, contra o agrupamento da Parte 2:
   Rand ajustado de 0.45 a 0.56 | média 0.51


In [30]:
train_clusters_per_seed = {}                                                                                                            # semente -> agrupamento só do treino
for seed in seeds_to_compare:
    train_clusters_per_seed[seed] = KMeans(n_clusters=number_of_clusters, random_state=seed, n_init=10).fit_predict(train_embeddings)   # só o treino, mudando a semente

agreement_between_train_seeds = []                                                                                                                 # o quanto duas sementes agrupam o treino do mesmo jeito
for first_seed in seeds_to_compare:
    for second_seed in seeds_to_compare:
        if first_seed < second_seed:
            agreement_between_train_seeds.append(adjusted_rand_score(train_clusters_per_seed[first_seed], train_clusters_per_seed[second_seed]))   # cada par de sementes uma vez

print("Só o treino, sementes diferentes comparadas entre si:")
print("   Rand ajustado de", round(min(agreement_between_train_seeds), 2), "a", round(max(agreement_between_train_seeds), 2), "| média", round(np.mean(agreement_between_train_seeds), 2))

Só o treino, sementes diferentes comparadas entre si:
   Rand ajustado de 0.39 a 0.61 | média 0.49


In [31]:
agreement_with_more_restarts = []                                                                                                        # o mesmo controle, deixando o K-Means tentar mais vezes por semente
for seed in range(10):
    clusters_with_more_restarts = KMeans(n_clusters=number_of_clusters, random_state=seed, n_init=50).fit_predict(activity_embeddings)   # 50 tentativas em vez de 10
    agreement_with_more_restarts.append(adjusted_rand_score(cluster_of_each_activity, clusters_with_more_restarts))                      # compara com o agrupamento da Parte 2

print("Mesmas 693 atividades, outra semente, com 50 tentativas por semente:")
print("   Rand ajustado de", round(min(agreement_with_more_restarts), 2), "a", round(max(agreement_with_more_restarts), 2))

Mesmas 693 atividades, outra semente, com 50 tentativas por semente:
   Rand ajustado de 0.49 a 0.56


**O que deu.** Com as mesmas 693 atividades e só a semente trocada, o Rand ajustado contra o agrupamento
da Parte 2 ficou entre 0,45 e 0,56 (média 0,51), praticamente o mesmo 0,46 que tínhamos atribuído à
retirada do teste. Comparando as sementes entre si só com o treino, ficou entre 0,39 e 0,61 (média
0,49), e deixar o K-Means tentar 50 vezes por semente não muda o quadro (de 0,49 a 0,56).

Logo, a mudança não veio de tirar o teste: nesta base o K-Means chega a divisões bem diferentes a cada
semente, porque os grupos se encostam.

O que entendemos é que a semente decide onde o K-Means finca os centros no começo. Fixar a semente
garante que a mesma conta dá o mesmo resultado, mas não que esse resultado seja o único possível, e
quando o conjunto de atividades muda, a mesma semente sorteia outros pontos de partida.

**Decisões que isso sustenta.** Não usamos a mesma semente para refazer os grupos no treino, e passamos
a tratar os 16 grupos da Parte 2 como uma divisão entre várias possíveis, o que combina com a silhueta
perto de zero que o parecer mandou olhar com cuidado. Os assuntos (os nomes) se mantêm em boa parte,
enquanto as fronteiras entre eles, não.

## Experimento 10. Reciclagem e Plantas sobrevivem a outra semente?

"Reciclagem e Sustentabilidade" e "Plantas e Alimentação" foram o resultado que mais nos animou na Parte
2, porque não existem nas listas dos sites, então depois do experimento 9 a pergunta ficou inevitável:
essas duas categorias existem mesmo, ou foram sorte da semente 42? Olhamos de três jeitos.

Primeiro, em 20 sementes, contamos em quantas cada uma vira um grupo próprio, que é quando um grupo novo
recebe pelo menos 60% das atividades antigas da categoria e pelo menos metade dele é dessas atividades.

In [32]:
# ====================================================================================================
#             EXPERIMENTO 10: EM QUANTAS SEMENTES RECICLAGEM E PLANTAS VIRAM GRUPO PRÓPRIO
# ====================================================================================================
def forms_own_cluster(old_clusters, new_clusters, old_cluster_number):
    is_old_member = old_clusters == old_cluster_number                                                                  # atividades que eram do grupo antigo
    new_clusters_of_old_members = pd.Series(new_clusters[is_old_member]).value_counts()                                 # para onde elas foram
    largest_new_cluster = new_clusters_of_old_members.index[0]                                                          # o grupo novo que mais recebeu
    share_of_old_members_received = new_clusters_of_old_members.iloc[0] / is_old_member.sum()                           # que fração das antigas foi para ele
    share_of_new_cluster_from_old = new_clusters_of_old_members.iloc[0] / (new_clusters == largest_new_cluster).sum()   # que fração dele veio das antigas
    return share_of_old_members_received >= 0.6 and share_of_new_cluster_from_old >= 0.5

recycling_cluster = 11   # grupo de Reciclagem e Sustentabilidade na Parte 2
plants_cluster = 13      # grupo de Plantas e Alimentação na Parte 2

valid_raw_embeddings = activity_embeddings[~has_broken_description]   # a base toda sem as 3 de descrição quebrada, sem normalizar (690)

times_recycling_in_whole_base = 0                                                                                                    # em quantas sementes Reciclagem vira grupo próprio, com a base toda
times_plants_in_whole_base = 0                                                                                                       # idem para Plantas
times_recycling_in_train = 0                                                                                                         # em quantas sementes Reciclagem vira grupo próprio, só com o treino
times_plants_in_train = 0                                                                                                            # idem para Plantas
for seed in range(20):
    clusters_of_whole_base = KMeans(n_clusters=number_of_clusters, random_state=seed, n_init=10).fit_predict(valid_raw_embeddings)   # o K-Means do notebook 05, com outra semente
    clusters_of_train_only = KMeans(n_clusters=number_of_clusters, random_state=seed, n_init=10).fit_predict(train_embeddings)       # o mesmo, só com o treino
    times_recycling_in_whole_base += forms_own_cluster(valid_clusters, clusters_of_whole_base, recycling_cluster)                    # soma 1 se virou grupo próprio
    times_plants_in_whole_base += forms_own_cluster(valid_clusters, clusters_of_whole_base, plants_cluster)                          # idem
    times_recycling_in_train += forms_own_cluster(old_clusters_of_train, clusters_of_train_only, recycling_cluster)                  # idem, no treino
    times_plants_in_train += forms_own_cluster(old_clusters_of_train, clusters_of_train_only, plants_cluster)                        # idem

print("Base toda: Reciclagem vira grupo próprio em", times_recycling_in_whole_base, "de 20 sementes | Plantas em", times_plants_in_whole_base, "de 20")
print("Só treino: Reciclagem vira grupo próprio em", times_recycling_in_train, "de 20 sementes | Plantas em", times_plants_in_train, "de 20")
print()
print("Na semente 42 só com o treino (experimento 9):")
for old_cluster_number in [recycling_cluster, plants_cluster]:
    new_clusters_of_old_members = pd.Series(new_clusters_of_train[old_clusters_of_train == old_cluster_number]).value_counts()   # para que grupos novos as atividades foram
    largest_new_cluster = new_clusters_of_old_members.index[0]                                                                   # o grupo novo que mais recebeu
    print("  ", name_of_each_cluster[old_cluster_number], "| espalhada por", len(new_clusters_of_old_members), "grupos novos | maior pedaço:", new_clusters_of_old_members.iloc[0], "atividades, num grupo de", (new_clusters_of_train == largest_new_cluster).sum())

Base toda: Reciclagem vira grupo próprio em 2 de 20 sementes | Plantas em 1 de 20
Só treino: Reciclagem vira grupo próprio em 0 de 20 sementes | Plantas em 2 de 20

Na semente 42 só com o treino (experimento 9):
   Reciclagem e Sustentabilidade | espalhada por 7 grupos novos | maior pedaço: 4 atividades, num grupo de 53
   Plantas e Alimentação | espalhada por 4 grupos novos | maior pedaço: 7 atividades, num grupo de 7


Depois, mudamos o número de grupos, para ver se com mais grupos as duas categorias teriam mais espaço
para aparecer.

In [33]:
# ----------------------------------------------------------------------------------------------------
#                           COM MAIS (OU MENOS) GRUPOS, ELAS APARECEM MAIS?
# ----------------------------------------------------------------------------------------------------
own_cluster_per_number_of_clusters = []                                                                                                                                      # uma linha por número de grupos
for tested_number_of_clusters in [12, 16, 20, 24, 30]:
    times_recycling = 0                                                                                                                                                      # em quantas sementes Reciclagem vira grupo próprio
    times_plants = 0                                                                                                                                                         # idem para Plantas
    for seed in range(10):
        tested_clusters = KMeans(n_clusters=tested_number_of_clusters, random_state=seed, n_init=10).fit_predict(valid_raw_embeddings)                                       # base toda, outro número de grupos
        times_recycling += forms_own_cluster(valid_clusters, tested_clusters, recycling_cluster)                                                                             # soma 1 se virou grupo próprio
        times_plants += forms_own_cluster(valid_clusters, tested_clusters, plants_cluster)                                                                                   # idem
    own_cluster_per_number_of_clusters.append({"number_of_clusters": tested_number_of_clusters, "recycling_out_of_10": times_recycling, "plants_out_of_10": times_plants})   # guarda as duas contagens

pd.DataFrame(own_cluster_per_number_of_clusters)

,number_of_clusters,recycling_out_of_10,plants_out_of_10
0,12,0,0
1,16,1,0
2,20,0,0
3,24,1,1
4,30,2,2


Por último, deixamos o K-Means de lado e olhamos as vizinhas, como no experimento 3: para cada atividade
de um grupo da Parte 2, contamos quantas das 10 atividades mais parecidas com ela são do mesmo grupo. Se
o assunto existe, as atividades dele ficam perto umas das outras, mesmo que o K-Means não consiga isolar
o grupo.

In [34]:
# ----------------------------------------------------------------------------------------------------
#                          AS VIZINHAS DE CADA ATIVIDADE SÃO DO MESMO GRUPO?
# ----------------------------------------------------------------------------------------------------
np.fill_diagonal(similarity_between_valid_activities, -1)                                                      # a atividade não pode ser vizinha dela mesma
closest_valid_neighbors = np.argsort(-similarity_between_valid_activities, axis=1)[:, :number_of_neighbors]    # as 10 mais parecidas, entre as 690 com descrição
neighbors_in_same_cluster = (valid_clusters[closest_valid_neighbors] == valid_clusters[:, None]).sum(axis=1)   # quantas das 10 são do mesmo grupo da atividade

neighbors_per_cluster = pd.DataFrame({"cluster": range(number_of_clusters)})                                                                                                           # uma linha por grupo
neighbors_per_cluster["category"] = neighbors_per_cluster["cluster"].map(name_of_each_cluster)                                                                                         # nome do grupo
neighbors_per_cluster["activities"] = [(valid_clusters == cluster_number).sum() for cluster_number in range(number_of_clusters)]                                                       # tamanho do grupo
neighbors_per_cluster["median_neighbors_in_same_cluster"] = [np.median(neighbors_in_same_cluster[valid_clusters == cluster_number]) for cluster_number in range(number_of_clusters)]   # mediana de vizinhas do mesmo grupo
neighbors_per_cluster["expected_by_chance"] = (number_of_neighbors * neighbors_per_cluster["activities"] / len(valid_clusters)).round(2)                                               # quantas viriam se as vizinhas fossem sorteadas

neighbors_per_cluster.sort_values("median_neighbors_in_same_cluster", ascending=False)

,cluster,category,activities,median_neighbors_in_same_cluster,expected_by_chance
5,5,Música,36,10.0,0.52
15,15,Esporte,50,10.0,0.72
10,10,Cinema,54,8.0,0.78
1,1,Atividade Física e Bem-Estar,54,8.0,0.78
7,7,Artesanato,40,7.0,0.58
9,9,Infância e Brincadeiras,48,6.5,0.70
8,8,Esporte,58,6.0,0.84
12,12,Artes Visuais,64,6.0,0.93
0,0,Literatura e Leitura,45,5.0,0.65
11,11,Reciclagem e Sustentabilidade,21,5.0,0.30


**O que deu.** Reciclagem e Plantas são frágeis para o K-Means. Com a base toda (690 atividades),
Reciclagem vira grupo próprio em 2 das 20 sementes e Plantas em 1, e só com o treino, Reciclagem em
nenhuma e Plantas em 2. Na semente 42 só com o treino, a do experimento 9, as atividades de Plantas se
dividem em 4 grupos, com um grupo só delas de 7 atividades, e as de Reciclagem se espalham por 7 grupos.
Mudar o número de grupos não resolve, já que de 12 a 30 grupos nenhuma das duas passa de 2 em 10
sementes.

As vizinhas contam outra história. Uma atividade de Reciclagem tem, na mediana, 5 das 10 vizinhas no
próprio grupo, quando o acaso daria 0,3, e uma de Plantas tem 3, contra 0,25 pelo acaso (para comparar,
Música tem 10 de 10 e um dos grupos de Shows tem 3). Logo, as atividades desses dois assuntos ficam bem
mais perto umas das outras do que o acaso explicaria.

A leitura que fazemos é que os dois assuntos existem nos textos, e o que falha é o K-Means conseguir
isolar cada um. Como são assuntos pequenos (21 e 17 atividades), achamos que ele acaba dividindo os
grupos grandes antes de abrir espaço para os pequenos, e a semente 42 da Parte 2 foi uma das poucas
rodadas em que os dois apareceram.

**Decisão que isso sustenta.** Não refazemos os 16 grupos nem os 13 nomes na Parte 3, porque refazer com
outra semente apagaria Reciclagem e Plantas, que pelas vizinhas parecem ser assuntos de verdade. Falta
só um jeito de recalcular esses mesmos grupos sem o teste, que é o próximo experimento.

## Experimento 11. Começar o K-Means pelos centros que já tínhamos

O experimento 9 mostrou que o problema da semente é o sorteio dos pontos de partida, então pensamos em
pular o sorteio: em vez de deixar o K-Means sortear, entregamos a ele os centros dos 16 grupos da Parte
2, calculados só com as atividades de treino, e deixamos ele seguir dali. O teste entra depois, indo
para o centro mais perto. Este experimento também foi feito sem as 3 de descrição quebrada.

In [35]:
# ====================================================================================================
#              EXPERIMENTO 11: K-MEANS COMEÇANDO NOS CENTROS DA PARTE 2 (SÓ COM O TREINO)
# ====================================================================================================
starting_centers = []                                                                                 # um ponto de partida por grupo
for cluster_number in range(number_of_clusters):
    starting_centers.append(train_embeddings[old_clusters_of_train == cluster_number].mean(axis=0))   # média das atividades de treino do grupo, sem normalizar
starting_centers = np.array(starting_centers)                                                         # lista vira matriz de 16 linhas

seeded_kmeans = KMeans(n_clusters=number_of_clusters, init=starting_centers, n_init=1, max_iter=300)   # sem sorteio: começa nos centros que entregamos
seeded_clusters_of_train = seeded_kmeans.fit_predict(train_embeddings)                                 # grupos do treino
seeded_clusters_of_test = seeded_kmeans.predict(test_embeddings)                                       # o teste vai para o centro mais perto

seeded_category_of_test = pd.Series(seeded_clusters_of_test).map(name_of_each_cluster).to_numpy()   # categoria do teste pelos grupos recalculados
kept_category_in_test = seeded_category_of_test == category_of_each_activity[is_test]               # ficou na mesma categoria da Parte 2?

print("Iterações até o K-Means parar:", seeded_kmeans.n_iter_)
print("Treino no mesmo grupo da Parte 2:", round(100 * (seeded_clusters_of_train == old_clusters_of_train).mean(), 1), "%")
print("Tamanho no treino: Reciclagem", (seeded_clusters_of_train == recycling_cluster).sum(), "| Plantas", (seeded_clusters_of_train == plants_cluster).sum())
print("Teste no mesmo grupo:", round(100 * (seeded_clusters_of_test == old_clusters_of_test).mean(), 1), "% | na mesma categoria:", kept_category_in_test.sum(), "de", is_test.sum(), "(", round(100 * kept_category_in_test.mean(), 1), "% )")

Iterações até o K-Means parar: 1
Treino no mesmo grupo da Parte 2: 100.0 %
Tamanho no treino: Reciclagem 16 | Plantas 13
Teste no mesmo grupo: 89.9 % | na mesma categoria: 126 de 139 ( 90.6 % )


In [36]:
# ----------------------------------------------------------------------------------------------------
#                           AS ATIVIDADES DE TESTE QUE MUDARAM DE CATEGORIA
# ----------------------------------------------------------------------------------------------------
test_activities_that_changed = activities.loc[is_test, ["title", "category"]][~kept_category_in_test].copy()   # título e categoria da Parte 2
test_activities_that_changed["category_with_train_only"] = seeded_category_of_test[~kept_category_in_test]     # categoria pelos grupos recalculados só com o treino
test_activities_that_changed

,title,category,category_with_train_only
2,JARDIM SECRETO – UM DIA PLANTANDO SONHOS COM E...,Infância e Brincadeiras,Literatura e Leitura
56,ESCREVENDO MINHA CANÇÃO – COM EQUIPE BIBLIOTECA,Artes Visuais,Literatura e Leitura
70,ANTES DA ANIMAÇÃO: BRINQUEDOS ÓPTICOS – COM EQ...,Artesanato,"Shows, Espetáculos e Performances"
72,DIA DO SACI | JAXY JATERÊ – CONFECÇÃO DE MARCA...,Artesanato,Literatura e Leitura
76,OFICINA CRIATIVA – VIRADA SUSTENTÁVEL,Reciclagem e Sustentabilidade,Artes Visuais
296,Natureza e Bem-Estar: Descubra Poços de Caldas...,Plantas e Alimentação,Esporte
369,Breve História do Tarot,"Shows, Espetáculos e Performances",Sociedade e Debates
380,Nhemboaty Nhemonguetá: Roda de escuta Guarani,Literatura e Leitura,Sociedade e Debates
513,Online - Tecnologia e o futuro das proteínas,Plantas e Alimentação,Sociedade e Debates
527,Arte e mídias digitais,Sociedade e Debates,Cultura e Identidade


**O que deu.** O K-Means parou na 1ª iteração, o que quer dizer que a divisão da Parte 2, recalculada só
com o treino, já é um ponto de equilíbrio dele: nenhuma atividade de treino quis trocar de grupo. Todo o
treino ficou no mesmo grupo, e os 16 grupos se mantiveram, inclusive Reciclagem (16 atividades de
treino) e Plantas (13).

No teste, 126 das 139 atividades (90,6%) ficaram na mesma categoria, e as 13 que mudaram são quase todas
atividades de beirada, como "Jardim Secreto" (de Infância para Literatura) e "Breve História do Tarot"
(de Shows para Sociedade e Debates).

**Decisão que isso sustenta.** No notebook 07, os grupos são recalculados só com o treino desse jeito,
começando nos centros da Parte 2. Como lá as 3 atividades de descrição quebrada ficam na base, o número
do teste pode mudar em uma atividade.

Juntando os experimentos 9, 10 e 11: o K-Means tem várias divisões possíveis nesta base, a da Parte 2 é
uma divisão em que ele se acomoda e que contém Reciclagem e Plantas, e as vizinhas indicam que esses
dois assuntos existem nos textos.